We aim to classify images of various fish species using a basic DNN model. The task involves processing image data as input, applying appropriate preprocessing steps, and developing a model to accurately predict the fish species from the images.

1. **Data Collection and Validation:**   
   The dataset used contains images of 9 different fish species. Dataset includes: gilt head bream, red sea bream, sea bass, red mullet, horse mackerel, black sea sprat, striped red mullet, trout, shrimp image samples. The images are resized to 590 x 445 pixels.   

2. **Exploratory Data Analysis and Visualization:**
   Before preprocessing, the image dimensions and color channels were analyzed, and necessary visualizations were created.

3. **Image Preprocessing:**
   Image data must be preprocessed before feeding it into the DNN.    
   TO DO: Apply the right image preprocessing 

In [ ]:
import os
import random
import shutil
import glob
import logging
from tqdm import tqdm
from itertools import compress
from typing import Optional, List, Tuple, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
from PIL import Image
import cv2 as cv
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.saving import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Layer, Flatten, LeakyReLU, ReLU, GlobalAveragePooling2D
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam ,RMSprop
from tensorflow.keras import saving

pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
plt.style.use("ggplot")
sns.set_palette(sns.diverging_palette(220, 20))

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "NOT AVAILABLE")

## 1. Data Collection:

### Comment utiliser le dataset Fish_Dataset

Le dataset est organise en dossiers. Chaque **nom de dossier** = **label (classe)**.

```
Fish_Dataset/
├── Black Sea Sprat/
│   └── Black Sea Sprat/   <- les images .png sont ici
├── Gilt-Head Bream/
│   └── Gilt-Head Bream/
...
```

**Pour Google Colab :**
```python
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
fish_dataset_directory = "/content/drive/MyDrive/dataset"
```


In [ ]:
# Adaptez ce chemin selon votre environnement
 Google Colab (decommentez si necessaire) :
 from google.colab import drive
 drive.mount('/content/drive', force_remount=True)
 fish_dataset_directory = "/content/drive/MyDrive/dataset"

In [ ]:
def find_image_classes(images_path: str) -> List[str]:
    """
    Find subdirectory names in the specified directory.
    """
    return [i for i in os.listdir(images_path) if os.path.isdir(os.path.join(images_path, i))]

In [ ]:
image_classes = find_image_classes(fish_dataset_directory)
image_classes

In [ ]:
def df_from_image_folders(images_path: str, extension: Optional[str] = "png") -> pd.DataFrame:
    """
    Create a DataFrame from image files in specified directories.
    Le LABEL = nom du dossier parent de chaque image.
    Exclut les dossiers contenant 'GT' (Ground Truth, pas utiles pour la classification).
    """
    label = []
    path = []
    image_files = glob.glob(os.path.join(images_path, "**", f"*.{extension.lower()}"), recursive=True)

    for file in image_files:
        dirpath = os.path.dirname(file)
        folder_name = os.path.basename(dirpath)
        # Exclure les dossiers GT (ground truth segmentation masks)
        if 'GT' not in folder_name:
            label.append(folder_name)
            path.append(file)

    class_dict = {"path": path, "label": label}
    return pd.DataFrame(class_dict)

In [ ]:
df = df_from_image_folders(fish_dataset_directory)
df.head()

### Explication : Label = Nom du dossier

Le **label** est automatiquement extrait depuis le **nom du dossier** parent de chaque image.

- Image path : `Fish_Dataset/Black Sea Sprat/Black Sea Sprat/00001.png`
- `os.path.basename(os.path.dirname(file))` retourne `"Black Sea Sprat"` = le **label**

Le DataFrame `df` contient 2 colonnes :
- `path` : chemin complet vers chaque image
- `label` : classe de l'image (nom du dossier = espece de poisson)

In [ ]:
def display_fish_from_each_class(df: pd.DataFrame, img_size: Tuple[int, int] = (224, 224)) -> None:
    """
    Displays one image from each unique class in the DataFrame.
    """
    plt.figure(figsize=(12, 12))
    
    for i, unique_label in enumerate(df["label"].unique()):
        plt.subplot(3, 3, i + 1)
        image_path = df[df["label"] == unique_label].iloc[0, 0]
        img = load_img(image_path, target_size=img_size)
        plt.imshow(img)
        plt.title(unique_label)
        plt.axis('off')

    plt.show()

In [ ]:
display_fish_from_each_class(df)

In [ ]:
def display_images_from_class(df: pd.DataFrame, class_name: str, num_images: int, img_size: Tuple[int, int] = (224, 224)) -> None:
    """
    Displays a specified number of images from a given class in the DataFrame.
    """
    images = df[df["label"] == class_name]["path"].iloc[:num_images]
    plt.figure(figsize=(12, 12))

    for i, image_path in enumerate(images):
        plt.subplot(4, 4, i + 1)
        plt.imshow(load_img(image_path, target_size=img_size))
        plt.title(image_path[-5:-4]) 
        plt.axis('off')

    plt.show()

In [ ]:
display_images_from_class(df, "Sea Bass", 8)

In [ ]:
df["label"].nunique()

In [ ]:
df[["label"]].value_counts()

## 2. Exploratory Data Analysis and Visualization:

### 2.1. Analysis of Image Sizes and Channels:

- The purpose of this study is to standardize images if they are different sizes. Using the created function below, it has been determined that all images have the same standard size.

In [ ]:
def compute_image_statistics_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes average width, height, and channel count for images listed in the DataFrame.
    """
    stats = []
    grouped = df.groupby('label')

    for label, group in grouped:
        widths = []
        heights = []
        channel_counts = []

        for _, row in group.iterrows():
            image_path = row['path']
            try:
                img = load_img(image_path)
                image_array = img_to_array(img)
                width, height = img.size
                widths.append(width)
                heights.append(height)
                channel_counts.append(image_array.shape[2])
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")
        
        if widths:  
            stats.append({
                'Fish Class': label,
                'Average Width': np.mean(widths),
                'Average Height': np.mean(heights),
                'Average Channels': np.mean(channel_counts),
                'Min Width': np.min(widths),
                'Max Width': np.max(widths),
                'Min Height': np.min(heights),
                'Max Height': np.max(heights)
            })

    return pd.DataFrame(stats)

In [ ]:
df_statistics = compute_image_statistics_from_df(df)
df_statistics

### 2.2. Visualization of the image rgb channels

In [ ]:
def display_rgb_channels(image_path: str) -> None:
    """
    Displays the individual RGB channels of an image.
    """
    image = Image.open(image_path)
    r, g, b = image.split()
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 12))
    axes[0].imshow(np.array(r), cmap="Reds")
    axes[0].set_title("Red Channel")
    axes[0].axis("off")
    axes[1].imshow(np.array(g), cmap="Greens")
    axes[1].set_title("Green Channel")
    axes[1].axis("off")
    axes[2].imshow(np.array(b), cmap="Blues")
    axes[2].set_title("Blue Channel")
    axes[2].axis("off")
    plt.show()

In [ ]:
sample_image = "./TP6/Fish_Dataset/Fish_Dataset/Sea Bass/Sea Bass/00026.png"
display_rgb_channels(sample_image)

### 2.3. Histogram of the each fish species images pixel distribution:

- As can be seen from the pixel distribution histogram graphs, each species has different color distributions

In [ ]:
def plot_average_pixel_distribution_from_df(df: pd.DataFrame, target_size: tuple = (150, 150)) -> None:
    """
    Plots average pixel value distribution for each class.
    """
    unique_labels = df['label'].unique()
    plt.figure(figsize=(15, 10))
 
    for i, label in enumerate(unique_labels):
        class_images = df[df['label'] == label]['path'].values
        pixel_values = []
        for image_path in class_images:
            img = load_img(image_path, target_size=target_size)
            pixel_values.append(img_to_array(img))
        
        pixel_values = np.array(pixel_values)
        avg_pixel_values = np.mean(pixel_values, axis=(0, 1, 2))  
        
        plt.subplot(3, 3, i + 1) 
        plt.hist(avg_pixel_values, bins=50, range=(0, 255), color='blue', alpha=0.7)
        plt.title(f'{label} Pixel Value Distribution')
        plt.xlabel('Pixel Value (0-255)')
        plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_average_pixel_distribution_from_df(df)

## 3. Image Preprocessing:

### Pourquoi pretraiter les images ?

| Etape | Description | Pourquoi ? |
|-------|-------------|------------|
| **Redimensionnement (Resize)** | Toutes les images -> 128x128 px | Le reseau attend une taille fixe en entree |
| **Normalisation** | Pixels de [0-255] -> [0.0-1.0] | Stabilise l'apprentissage, evite les gradients explosifs |
| **Conversion RGB** | Forcer le mode couleur RGB | Garantit 3 canaux (R, G, B) pour chaque image |

**Choix : Sans extraction de caracteristiques**
- On passe l'image brute aplatie (flatten = vecteur de pixels) directement au reseau DNN
- Le DNN apprend ses propres features directement depuis les pixels
- Avantage : simple, pas besoin de definir manuellement les descripteurs
- L'output de flatten est directement un vecteur numerique pret pour le DNN

In [ ]:
def load_image(image_path: str) -> Image.Image:
    """
    Load an image in RGB format from the given path.
    """
    image = Image.open(image_path).convert("RGB")
    return image

def load_images_from_df(df: pd.DataFrame) -> List[Image.Image]:
    """
    Load images from a DataFrame containing image paths.
    """
    images = [load_image(image_path) for image_path in tqdm(df["path"].values.tolist(), total=len(df))]
    return images

In [ ]:
loaded_images = load_images_from_df(df)

In [ ]:
def plot_image(image) -> None:
    """
    Display an image using matplotlib with the axis turned off.
    """
    plt.imshow(image)
    plt.axis("off")
    plt.show()

In [ ]:
plot_image(loaded_images[20])

In [ ]:
def preprocess(pil_image: Image.Image, target_size: Tuple[int, int] = (128, 128)) -> np.ndarray:
    """
    Preprocess a PIL image for DNN input.

    Steps appliques :
    1. Resize a target_size (128x128 par defaut) -> taille uniforme pour le reseau
    2. Convert to RGB -> garantit 3 canaux couleur
    3. Convert to numpy array
    4. Normalize pixel values [0-255] -> [0.0-1.0] -> stabilise l'entrainement

    Parameters
    ----------
    pil_image : Image.Image
        The input PIL image to preprocess.
    target_size : Tuple[int, int]
        Desired output size. Default: (128, 128)
        
    Returns
    -------
    np.ndarray
        Preprocessed image as numpy array of shape (128, 128, 3), values in [0.0, 1.0]
    """
    # Etape 1 : Redimensionner l'image a la taille cible (128x128)
    # LANCZOS = meilleur filtre de sous-echantillonnage (preserve les details)
    img = pil_image.resize(target_size, Image.LANCZOS)

    # Etape 2 : S'assurer que l'image est en mode RGB (3 canaux)
    # Certaines images peuvent etre en mode L (grayscale) ou RGBA
    img = img.convert("RGB")

    # Etape 3 : Convertir en tableau numpy float32
    img_array = np.array(img, dtype=np.float32)

    # Etape 4 : Normalisation [0-255] -> [0.0-1.0]
    # Tres important pour la convergence du reseau de neurones
    img_array = img_array / 255.0

    return img_array

In [ ]:
preprocessed_image = preprocess(loaded_images[20])
print(f"Shape apres pretraitement: {preprocessed_image.shape}")
print(f"Valeur min: {preprocessed_image.min():.3f}, Valeur max: {preprocessed_image.max():.3f}")
plot_image(preprocessed_image)

**Applying preprocessing to The All Images and Saving Them to Disk**

In [ ]:
def process_and_save_images(df: pd.DataFrame, output_base_dir: str) -> None:
    """
    Load, process, and save images organized by labels into respective directories.
    
    Pour chaque image du DataFrame :
    1. Charger l'image depuis son chemin
    2. Appliquer le pretraitement (resize + normalize)
    3. Sauvegarder dans output_base_dir/label/nom_fichier.png
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les colonnes 'path' et 'label'.
    output_base_dir : str
        Repertoire de base ou les images seront sauvegardees, organisees par label.
    """
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing & saving images"):
        image_path = row['path']
        label = row['label']

        # Creer le dossier pour ce label s'il n'existe pas
        label_dir = os.path.join(output_base_dir, label)
        os.makedirs(label_dir, exist_ok=True)

        # Charger l'image
        pil_image = load_image(image_path)

        # Appliquer le pretraitement
        preprocessed = preprocess(pil_image)

        # Reconvertir en image PIL pour sauvegarde (pixels [0-1] -> [0-255])
        img_to_save = Image.fromarray((preprocessed * 255).astype(np.uint8))

        # Sauvegarder avec le meme nom de fichier
        filename = os.path.basename(image_path)
        save_path = os.path.join(label_dir, filename)
        img_to_save.save(save_path)

In [ ]:
import os
output_path = "./TP6/Fish_Dataset/Fish_Dataset/preprocessed_images"
os.makedirs(output_path, exist_ok=True)

In [ ]:
process_and_save_images(df, output_path)

**Control**

- All processed images have been controlled using `df_from_image_folders` function that i created before

In [ ]:
preprocessed_images = "./TP6/Fish_Dataset/Fish_Dataset/preprocessed_images"
df_preprocessed = df_from_image_folders(preprocessed_images)
df_preprocessed.head()

In [ ]:
df_preprocessed["label"].value_counts()

In [ ]:
display_fish_from_each_class(df_preprocessed)

## 4. Neural Network (NN) Model Creation:

### 4.1. Splitting the Fish_Dataset into Train, Validation, and Test Sets.

10% of the dataset was allocated for validation and another 10% for testing. The validation set was used during model training, while the test set was reserved for evaluating the model's performance after training was completed.

In [ ]:
def copy_images(subset_df: pd.DataFrame, target_dir: str) -> None:
    for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Copying to {target_dir}"):
        shutil.copy2(row['path'], target_dir)


def split_data_into_train_val_test(
    df_processed: pd.DataFrame, 
    output_dir: str, 
    train_ratio: float = 0.8, 
    val_ratio: float = 0.1, 
    test_ratio: float = 0.1,
    random_seed: int = 42 
) -> pd.DataFrame:
    np.random.seed(random_seed)
    df_processed = shuffle(df_processed).reset_index(drop=True)
    class_summary = {'Class': [], 'Train Count': [], 'Validation Count': [], 'Test Count': []}
    
    for class_name, class_group in df_processed.groupby('label'):
        total_images = len(class_group)
        train_count = int(total_images * train_ratio)
        val_count = int(total_images * val_ratio)
        test_count = total_images - train_count - val_count

        class_summary['Class'].append(class_name)
        class_summary['Train Count'].append(train_count)
        class_summary['Validation Count'].append(val_count)
        class_summary['Test Count'].append(test_count)

        train_df = class_group[:train_count]
        val_df = class_group[train_count:train_count + val_count]
        test_df = class_group[train_count + val_count:]
        
        train_dir = os.path.join(output_dir, 'train', class_name)
        val_dir = os.path.join(output_dir, 'validation', class_name)
        test_dir = os.path.join(output_dir, 'test', class_name)
        
        for dir_path in [train_dir, val_dir, test_dir]:
            if not os.path.exists(dir_path):
                os.makedirs(dir_path)
        
        copy_images(train_df, train_dir)
        copy_images(val_df, val_dir)
        copy_images(test_df, test_dir)
    
    return pd.DataFrame(class_summary)

In [ ]:
model_images_output = "./TP6/Fish_Dataset/Fish_Dataset/model_dataset"
summary_df = split_data_into_train_val_test(df_preprocessed, model_images_output)
summary_df

### 4.2. Data Augmentation

**Data Augmentation Steps:**

- `Normalization`: Rescaling pixel values from the range of 0-255 to 0-1.
- `Random Rotation`: Rotating images up to 40 degrees randomly.
- `Random Width and Height Shift`: Shifting images randomly by 20% of their width and height.
- `Random Shear`: Applying random shearing transformations with a shear intensity of 0.2.
- `Random Zoom`: Zooming in on images randomly by 20%.
- `Horizontal Flip`: Flipping images horizontally.
- `Fill Method`: Using the 'nearest' fill mode for newly created pixels during transformations.

### Pourquoi ces augmentations pour les poissons ?

| Augmentation | Justification |
|---|---|
| **Rotation (40 deg)** | Un poisson peut etre photographie sous differents angles |
| **Horizontal Flip** | Un poisson nage aussi bien a gauche qu'a droite (realiste!) |
| **Width/Height Shift** | Le poisson peut etre decentre dans la photo |
| **Zoom** | Differentes distances de capture (proche/loin) |
| **Shear** | Legere deformation due a l'angle de la camera sous l'eau |

**Ce qu'on n'applique PAS** : `Vertical Flip` -> un poisson a l'envers n'est pas naturel !

**Objectif** : Eviter l'overfitting en montrant des variantes realistes de chaque image au modele.

In [ ]:
train_dir = './TP6/Fish_Dataset/Fish_Dataset/model_dataset/train'  
validation_dir = './TP6/Fish_Dataset/Fish_Dataset/model_dataset/validation'

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,          
    rotation_range=40,       
    width_shift_range=0.2,    
    height_shift_range=0.2,
    shear_range=0.2,          
    zoom_range=0.2,           
    horizontal_flip=True,
    # PAS de vertical_flip : un poisson a l'envers n'est pas realiste
    fill_mode='nearest'   
)

val_datagen = ImageDataGenerator(rescale=1./255)  # Pas d'augmentation sur validation

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150), 
    batch_size=32,
    class_mode='categorical'  # CORRECTION: 9 classes -> categorical (pas binary!)
)

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),  
    batch_size=32,
    class_mode='categorical'  # CORRECTION: 9 classes -> categorical
)

class_indices = train_generator.class_indices
print("Class labels:", class_indices)

**Data augmentation Control**

In [ ]:
batch = next(train_generator)
images, labels = batch
num_images = min(5, len(images))

plt.figure(figsize=(10, 10))
for i in range(num_images):
    plt.subplot(1, 5, i + 1)
    plt.imshow(images[i]) 
    plt.title(f"Label: {np.argmax(labels[i])}")  # categorical -> argmax pour afficher l'index
    plt.axis('off') 
plt.show()

### 4.3. Creation of the ANN Architecture

**Hyperparametres choisis :**

```
- Number of Layers    : 3 couches cachees Dense
- Number of Neurons   : 512 -> 256 -> 128 (diminution progressive)
- Batch size          : 32
- Activation Function : ReLU (couches cachees) + Softmax (sortie 9 classes)
- Optimization        : Adam
- Learning Rate       : 0.001
- Image size          : 128 x 128 x 3
```

**Architecture DNN :**
```
Input (128x128x3)
  -> Flatten (vecteur 49152 valeurs)
  -> Dense(512) + BatchNorm + ReLU + Dropout(0.4)
  -> Dense(256) + BatchNorm + ReLU + Dropout(0.3)
  -> Dense(128) + BatchNorm + ReLU + Dropout(0.2)
  -> Dense(9, softmax)  <- 9 classes de poissons
```

**Techniques anti-overfitting :**
- **BatchNormalization** : met la moyenne a 0 et l'ecart-type a 1 -> stabilise l'entrainement
- **Dropout** : desactive aleatoirement des neurones -> evite la dependance excessive
- **EarlyStopping** : arrete si pas d'amelioration apres 6 epochs
- **ReduceLROnPlateau** : reduit le learning rate si bloque (patience=4)
- **Data Augmentation** : variantes realistes des images

In [ ]:
@saving.register_keras_serializable()
class CustomDNN(Model):
    def __init__(self, input_shape=(128, 128, 3), num_classes=9, name="custom_ann", **kwargs):
        super(CustomDNN, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self._input_shape = input_shape

        # Aplatir l'image : (128, 128, 3) -> vecteur de 49152 valeurs
        self.flatten = Flatten()

        # Couche cachee 1 : 512 neurones
        self.dense1 = Dense(512)
        self.bn1 = BatchNormalization()  # Normalise : moyenne=0, ecart-type=1
        self.relu1 = ReLU()              # Activation non-lineaire
        self.dropout1 = Dropout(0.4)    # Desactive 40% des neurones (anti-overfitting)

        # Couche cachee 2 : 256 neurones
        self.dense2 = Dense(256)
        self.bn2 = BatchNormalization()
        self.relu2 = ReLU()
        self.dropout2 = Dropout(0.3)

        # Couche cachee 3 : 128 neurones
        self.dense3 = Dense(128)
        self.bn3 = BatchNormalization()
        self.relu3 = ReLU()
        self.dropout3 = Dropout(0.2)

        # Couche de sortie : 9 classes -> softmax (probabilites)
        self.classifier = Dense(num_classes, activation="softmax", dtype="float32")  

    def call(self, inputs, training=False):
        # Flatten : image -> vecteur 1D
        x = self.flatten(inputs)
        
        # Couche 1
        x = self.dense1(x)
        x = self.bn1(x, training=training)       # training=True: utilise stats du batch
        x = self.relu1(x)
        x = self.dropout1(x, training=training)  # Dropout actif seulement en training

        # Couche 2
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        x = self.relu2(x)
        x = self.dropout2(x, training=training)

        # Couche 3
        x = self.dense3(x)
        x = self.bn3(x, training=training)
        x = self.relu3(x)
        x = self.dropout3(x, training=training)

        # Sortie : probabilites pour chaque classe
        outputs = self.classifier(x)  
        return outputs

    def get_config(self):
        config = super(CustomDNN, self).get_config()
        config.update({
            "input_shape": self._input_shape, 
            "num_classes": self.num_classes
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def train(self, train_dir, val_dir, batch_size=32, epochs=10):
        
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            # Augmentations choisies pour leur realisme pour les poissons
            rotation_range=40,       # Poisson photographie sous differents angles
            width_shift_range=0.2,   # Poisson decentre horizontalement
            height_shift_range=0.2,  # Poisson decentre verticalement
            shear_range=0.2,         # Legere deformation angulaire
            zoom_range=0.2,          # Differentes distances de capture
            horizontal_flip=True,    # Poisson nage a gauche ou a droite (realiste)
            # PAS de vertical_flip : un poisson a l'envers n'existe pas dans la realite
            fill_mode='nearest'      # Remplir les pixels vides par les voisins
        )
        
        # IMPORTANT : pas d'augmentation sur les donnees de validation !
        validation_datagen = ImageDataGenerator(rescale=1./255)

        target_size = self._input_shape[:2]
        
        train_generator = train_datagen.flow_from_directory(
            train_dir,
            target_size=target_size,
            batch_size=batch_size,
            class_mode="categorical",  # 9 classes -> one-hot encoding
            color_mode="rgb",
            shuffle=True,
        )
        
        validation_generator = validation_datagen.flow_from_directory(
            val_dir,
            target_size=target_size,
            batch_size=batch_size,
            class_mode="categorical",
            color_mode="rgb",
            shuffle=False,
        )
        
        # Callback 1 : Sauvegarder le meilleur modele (val_loss minimale)
        os.makedirs("./TP6/Fish_Dataset/Fish_Dataset/models", exist_ok=True)
        checkpoint_val_loss = ModelCheckpoint(
            filepath="./TP6/Fish_Dataset/Fish_Dataset/models/best_model.keras",  
            monitor="val_loss",
            save_best_only=True,  
            save_weights_only=False,
            mode="min",
            verbose=1
        )

        # Callback 2 : Arreter l'entrainement si pas d'amelioration apres 6 epochs
        early_stopping_callback = EarlyStopping(
            monitor="val_loss",
            patience=6,  
            verbose=1,
            restore_best_weights=True  # Restaurer les meilleurs poids automatiquement
        )

        # Callback 3 : Reduire le learning rate x0.3 si bloque apres 4 epochs
        lr_callback = ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.3, 
            patience=4,
            min_lr=0.00001,
            verbose=1) 
        
        # Adam avec learning rate 0.001 (plus stable que 0.01)
        optimizer = Adam(learning_rate=0.001)
 
        self.compile(optimizer=optimizer,
                     loss="categorical_crossentropy",  # Pour classification multi-classe
                     metrics=["accuracy"])
        
        history = self.fit(
            train_generator,
            epochs=epochs,
            validation_data=validation_generator,
            callbacks=[checkpoint_val_loss, early_stopping_callback, lr_callback],
        )
        
        return history

In [ ]:
train_dir = "./TP6/Fish_Dataset/Fish_Dataset/model_dataset/train"
val_dir = "./TP6/Fish_Dataset/Fish_Dataset/model_dataset/validation"

model = CustomDNN(input_shape=(128, 128, 3), num_classes=9)
result = model.train(train_dir=train_dir, val_dir=val_dir, batch_size=32, epochs=200)

In [ ]:
model = load_model("./TP6/Fish_Dataset/Fish_Dataset/models/best_model.keras")
model.summary()

**Visualize with History**

In [ ]:
def plot_training_history(result: Dict[str, list]) -> None:
    """
    Plots the training and validation loss and accuracy graphs.
    """
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(result.history['loss'], label='Train Loss')
    plt.plot(result.history['val_loss'], label='Validation Loss')
    plt.title('Loss Graph')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(result.history['accuracy'], label='Train Accuracy')
    plt.plot(result.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy Graph')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_history(result)

### 4.4. Model Evaluation

The following tasks have been carried out:

- After training is completed, the model's performance is evaluated on the test set.
  
- The performance of the model is analyzed using metrics such as classification accuracy, confusion matrix, accuracy, precision, and recall.
  
- Errors are examined by visualizing the examples that the model misclassified.

**Evaluation Metrics**

**Precision:**
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall:**
$$\text{Recall} = \frac{TP}{TP + FN}$$

**Accuracy:**
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

---

**Class-Level Insights:**

- `Class 0 (Black Sea Sprat):` 
- `Class 1 (Gilt-Head Bream):` 
- `Class 2 (Horse Mackerel):` 
- `Class 3 (Red Mullet):` 
- `Class 4 (Red Sea Bream):` 
- `Class 5 (Sea Bass):` 
- `Class 6 (Shrimp):` 
- `Class 7 (Striped Red Mullet):` 
- `Class 8 (Trout):` 

---

**Conclusion:**


In [ ]:
def evaluate_model(
    model, 
    test_dir: str, 
    target_size: tuple = (128, 128), 
    batch_size: int = 32
) -> None:
    """
    Evaluate the model on the test dataset and display performance metrics.
    """
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='categorical', 
        shuffle=False
    )

    predictions = model.predict(test_generator)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = test_generator.classes

    accuracy = accuracy_score(true_classes, predicted_classes)
    precision = precision_score(true_classes, predicted_classes, average='macro')  
    recall = recall_score(true_classes, predicted_classes, average='macro') 

    print("Accuracy:", accuracy)
    print("Precision (macro):", precision)
    print("Recall (macro):", recall)

    conf_matrix = confusion_matrix(true_classes, predicted_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

    print(classification_report(true_classes, predicted_classes))
    
    return test_generator, predicted_classes


In [ ]:
test_directory = "./TP6/Fish_Dataset/Fish_Dataset/model_dataset/test"
test_generator, predicted_classes = evaluate_model(model, test_directory)

**Analyze Misclassifications**

In [ ]:
def analyze_and_visualize_misclassifications(test_generator, predicted_classes, num_samples=10) -> None:
    """
    Analyze and visualize misclassified images from the test dataset.
    """
    test_generator.reset()

    df = pd.DataFrame({
        'filename': test_generator.filenames,
        'predict': predicted_classes,  
        'y': test_generator.classes
    })
    misclassified = df[df['y'] != df['predict']]
    total_misclassified = misclassified.shape[0]
    
    print(f'Total misclassified images: {total_misclassified}')
    print("\nMisclassified Images:")
    print(misclassified)

    if total_misclassified > 0:
        samples_to_display = min(num_samples, total_misclassified)
        misclassified_samples = misclassified.sample(samples_to_display)
        
        cols = 5
        rows = (samples_to_display + cols - 1) // cols 
        fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
        
        for ax, (_, row) in zip(axes.flatten(), misclassified_samples.iterrows()):
            img_path = test_generator.directory + '/' + row['filename']
            img = plt.imread(img_path)
            ax.imshow(img)
            ax.set_title(f'True: {row["y"]}, Pred: {row["predict"]}')
            ax.axis('off')

        for i in range(samples_to_display, len(axes.flatten())):
            axes.flatten()[i].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
analyze_and_visualize_misclassifications(test_generator, predicted_classes, num_samples=10)

### 4.5. **Prediction**

A random image for each fish in the test folder has been selected and compiled into a list and it was observed that the model successfully made accurate predictions for fish species.

In [ ]:
def predict_single_image(model, image_path, test_generator):
    """
    Predict the class of a single image using a trained model.
    """
    image = load_img(image_path, target_size=(128, 128))
    plt.imshow(image)
    plt.title(f"Selected Image: {os.path.basename(image_path)}")
    plt.axis('off')
    plt.show()
   
    image_array = img_to_array(image) / 255.0 
    image_array = np.expand_dims(image_array, axis=0)  

    predictions = model.predict(image_array)
    predicted_index = np.argmax(predictions, axis=1)[0] 

    class_indices = test_generator.class_indices
    class_labels = {v: k for k, v in class_indices.items()} 

    predicted_label = class_labels[predicted_index]
    confidence = predictions[0][predicted_index]

    print(f"Predicted Label: {predicted_label} (Confidence: {confidence:.2f})")


**Prediction list for each class from the test image directory**

In [ ]:
test_folder = "./TP6/Fish_Dataset/Fish_Dataset/model_dataset/test" 
test_df = df_from_image_folders(test_folder)
prediction_list = test_df.groupby("label")["path"].sample(1).to_list()

In [ ]:
for image in prediction_list:
    class_name = os.path.basename(os.path.dirname(image))
    predict_single_image(model, image, test_generator)
    print(f"Actual label: {class_name}")
    print(50 * " ")
    print(50 * "*")